# Area Analysis

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pyrootutils
from xenocanto3 import Quality

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [3]:
LAT, LON = 48.8566, 2.3522            # Paris
RADIUS_KM = 50
SPECIES_NAMES = ["Emberiza calandra","Hippolais polyglotta","Regulus ignicapilla"]
MIN_QUALITY = Quality.B
MIN_RECORDINGS = 10                   # drop species with fewer than this many recordings in the area from the taxonomy breakdown

# BirdNET pass
MAX_BIRDNET_RECORDINGS = 500          # cap downloads 
BIRDNET_MIN_CONFIDENCE = 0.85      

# Bounding box

In [ ]:
from building.analysis import bbox_from_radius

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
print(f"bbox (lat_min, lon_min, lat_max, lon_max): {tuple(round(x, 4) for x in bbox)}")
print(f"lat span: {bbox[2] - bbox[0]:.3f}°   lon span: {bbox[3] - bbox[1]:.3f}°")

# Per-species global heatmap

In [ ]:
from building.analysis import fetch_species_global, recordings_to_df, build_heatmap, ANALYSIS_DIR

global_recs = await fetch_species_global(SPECIES_NAMES, min_quality=MIN_QUALITY)
per_species_df = {name: recordings_to_df(recs) for name, recs in global_recs.items()}

for name, df in per_species_df.items():
    print(f"{name:30s} {len(df):5d} recordings (with coords: {df['lat'].notna().sum()})")

heatmap = build_heatmap(per_species_df, center=(LAT, LON), bbox=bbox)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
heatmap.save(str(ANALYSIS_DIR / "heatmap.html"))
heatmap

Emberiza calandra                229 recordings (with coords: 228)
Hippolais polyglotta             401 recordings (with coords: 398)
Regulus ignicapilla              633 recordings (with coords: 626)


# Recordings inside the area

In [ ]:
from building.analysis import fetch_area_recordings

area_recs = await fetch_area_recordings(bbox, min_quality=MIN_QUALITY)
area_df = recordings_to_df(area_recs)
print(f"{len(area_df)} recordings, {area_df['scientific_name'].nunique()} unique species")

764 recordings, 145 unique species


# Are the target species present in the area?

In [ ]:
from building.analysis import presence_in_area

presence_in_area(SPECIES_NAMES, area_df)

,scientific_name,recordings_in_area,qualities,mean_length_sec
0,Emberiza calandra,2,A,59.0
1,Hippolais polyglotta,26,A,92.5
2,Regulus ignicapilla,20,A,85.8


# Taxonomy breakdown of the area

In [ ]:
from building.analysis import taxonomy_breakdown

tax = taxonomy_breakdown(area_df, min_recordings=MIN_RECORDINGS)
totals = tax["totals"].iloc[0]
print(
    f"{totals['n_recordings']} recordings  |  "
    f"{totals['n_species']} species across {totals['n_genera']} genera, "
    f"{totals['n_families']} families, {totals['n_orders']} orders"
    f"  (species with ≥ {MIN_RECORDINGS} recordings)"
)
if totals["n_unmatched_in_taxonomy"]:
    print(f"({totals['n_unmatched_in_taxonomy']} recordings could not be matched to the eBird taxonomy)")

In [ ]:
tax["by_order"]

,order,species,recordings
0,Passeriformes,17,275
1,NaN,3,41
2,Strigiformes,2,30
3,Piciformes,1,22
4,Psittaciformes,1,21
5,Coraciiformes,1,14


In [ ]:
tax["by_family"].head(20)

,order,family,species,recordings
0,Passeriformes,Sylviidae (Sylviid Warblers and Allies),3,48
1,NaN,NaN,3,41
2,Passeriformes,Acrocephalidae (Reed Warblers and Allies),2,39
3,Passeriformes,Muscicapidae (Old World Flycatchers),2,30
4,Passeriformes,Locustellidae (Grassbirds and Allies),2,28
5,Passeriformes,Motacillidae (Wagtails and Pipits),1,26
6,Piciformes,Picidae (Woodpeckers),1,22
7,Passeriformes,"Paridae (Tits, Chickadees, and Titmice)",1,22
8,Psittaciformes,Psittaculidae (Old World Parrots),1,21
9,Passeriformes,Regulidae (Kinglets),1,20


In [ ]:
tax["by_genus"].head(20)

,family,genus,species,recordings
0,NaN,NaN,3,41
1,Sylviidae (Sylviid Warblers and Allies),Curruca,2,34
2,Locustellidae (Grassbirds and Allies),Locustella,2,28
3,Acrocephalidae (Reed Warblers and Allies),Hippolais,1,26
4,Motacillidae (Wagtails and Pipits),Anthus,1,26
5,Picidae (Woodpeckers),Dendrocoptes,1,22
6,"Paridae (Tits, Chickadees, and Titmice)",Parus,1,22
7,Psittaculidae (Old World Parrots),Psittacula,1,21
8,Regulidae (Kinglets),Regulus,1,20
9,Certhiidae (Treecreepers),Certhia,1,18


# Co-occurring species

In [ ]:
from building.analysis import co_occurrence_counts

co_occurrence_counts(area_df, top_n=30)

,species,count
0,Fringilla coelebs,11
1,Phylloscopus collybita,10
2,Corvus corone,8
3,Troglodytes troglodytes,6
4,Turdus merula,6
5,Psittacula krameri,5
6,Sylvia atricapilla,4
7,Poecile palustris,4
8,Parus major,4
9,Erithacus rubecula,3


# Annotations

In [ ]:
from building.analysis import annotation_summary

ann = annotation_summary(area_df)
print(f"{len(ann['annotations'])} annotations across {len(ann['by_species'])} species")
ann["by_species"].head(20)

0 annotations across 0 species


,annotated_species,count


In [ ]:
ann["annotations"].head(20)

""


# BirdNET pass on area recordings

In [ ]:
from building.analysis import area_audio_cache_dir, birdnet_area_analysis

bn = await birdnet_area_analysis(
    area_df,
    max_recordings=MAX_BIRDNET_RECORDINGS,
    min_confidence=BIRDNET_MIN_CONFIDENCE,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
det = bn["detections"]
print(
    f"{len(det)} detections across {det['xc_id'].nunique()} recordings  |  "
    f"{bn['by_species'].shape[0]} unique species detected  |  "
    f"{bn['hidden'].shape[0]} species not declared in XC `also`"
)

# All area species, by source

In [ ]:
from building.analysis import combined_species_table

combined = combined_species_table(area_df, bn["detections"], min_recordings=MIN_RECORDINGS)
print(f"{len(combined)} species ( {MIN_RECORDINGS} recordings of any source)")
combined

38 species ( 10 recordings of any source)


,scientific_name,xc_primary,xc_also,birdnet_only,total
0,Hippolais polyglotta,26,0,5,31
1,Psittacula krameri,21,5,3,29
2,Parus major,22,4,3,29
3,Anthus spinoletta,26,0,2,28
4,Dendrocoptes medius,22,0,3,25
5,Curruca communis,22,0,2,24
6,Fringilla coelebs,10,11,2,23
7,Sylvia atricapilla,14,4,4,22
8,Regulus ignicapilla,20,0,2,22
9,Certhia brachydactyla,18,1,3,22


# Longest recordings in the area

In [ ]:
from building.analysis import longest_recordings

longest_recordings(area_df, center=(LAT, LON), top_n=10)

,id,scientific_name,en,length_seconds,distance_km,q,loc,date
0,767964,Anthus spinoletta,Water Pipit,1362,24.36,A,"Gif-sur-Yvette, Essonne, Île-de-France",2022-11-19
1,549540,Sylvia borin,Garden Warbler,1271,13.36,A,"Ville-d'Avray, Hauts-de-Seine, Île-de-France",2020-04-22
2,475269,Dendrocopos major,Great Spotted Woodpecker,1049,12.67,A,"Ville-d'Avray, Hauts-de-Seine, Île-de-France",2019-05-18
3,560923,Acrocephalus palustris,Marsh Warbler,835,23.14,A,"Gif-sur-Yvette, Essonne, Île-de-France",2020-05-22
4,535386,Athene noctua,Little Owl,821,25.58,A,"Villepreux, Yvelines, Île-de-France",2020-03-17
5,980020,Phoenicurus phoenicurus,Common Redstart,736,53.88,A,"La Roche qui pleure (near Fontainebleau), Sei...",2024-05-26
6,1102933,Locustella naevia,Common Grasshopper Warbler,717,31.42,A,"Arrondissement de Provins (near Favières), Se...",2026-04-19
7,476821,Tyto alba,Western Barn Owl,696,32.69,A,"Senlisse, Yvelines, Île-de-France",2019-05-26
8,994541,Turdus merula,Common Blackbird,669,31.68,A,"Arrondissement de Provins (near Favières), Se...",2025-04-20
9,560922,Acrocephalus palustris,Marsh Warbler,526,23.14,A,"Gif-sur-Yvette, Essonne, Île-de-France",2020-05-22
